# 🎵 Comparaison des 4 modèles Demucs

> **Notebook d'analyse comparative** — Séparation de sources audio avec Demucs

Ce notebook compare en profondeur les 4 variantes du modèle Demucs :

| Modèle | Description |
|--------|-------------|
| **htdemucs** | Hybrid Transformer Demucs (baseline) |
| **htdemucs_ft** | HTDemucs fine-tuné sur MUSDB-HQ |
| **mdx_extra** | MDX-Net (architecture convolutive) |
| **mdx_extra_q** | MDX-Net quantifié (plus léger) |

### Plan d'analyse
1. ⚙️ Configuration & chargement du fichier audio test
2. ⏱️ Benchmarking des temps d'inférence
3. 📊 Métriques objectives (SDR / SIR / SAR / ISR) avec vérité terrain
4. 🌊 Analyse des formes d'onde
5. 🔊 Spectrogrammes comparatifs
6. 📐 Métriques spectrales avancées (centroïde, rolloff, flux, MFCC distance)
7. ⚡ SI-SNR & énergie par bande de fréquences
8. 🕸️ Vue synthèse radar
9. 🏆 Verdict final


---
## ⚙️ 1. Configuration & Imports

In [ ]:
import sys
import os
import warnings
warnings.filterwarnings('ignore')

# ─── Ajout du projet au path ───────────────────────────────────────────────────
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# ─── Imports standard ─────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from pathlib import Path
from IPython.display import Audio, display, Markdown
import time
import pandas as pd

# ─── Style global dark mode ───────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor':  '#0d1117',
    'axes.facecolor':    '#0d1117',
    'text.color':        'white',
    'axes.labelcolor':   'white',
    'xtick.color':       'white',
    'ytick.color':       'white',
    'axes.edgecolor':    '#333333',
    'grid.color':        '#333333',
    'legend.facecolor':  '#1a1a2e',
    'legend.edgecolor':  '#444444',
    'font.family':       'DejaVu Sans',
    'font.size':         11,
})

# ─── Imports du projet ────────────────────────────────────────────────────────
from src.music_separation import (
    AudioSeparator, AudioEvaluator, Visualizer, audio_utils, analysis_tools
)
from src.music_separation.config import SUPPORTED_MODELS, DEFAULT_SAMPLE_RATE
from src.music_separation.analysis_tools import (
    Timer, MODEL_COLORS, MODEL_LABELS, STEM_NAMES,
    compute_snr, compute_si_snr, compute_rms_energy,
    compute_spectral_centroid, compute_spectral_rolloff, compute_spectral_flux,
    compute_mfcc_distance, compute_frequency_band_energy,
    plot_metrics_bar_per_stem, plot_waveforms_comparison, plot_spectrograms_grid,
    plot_frequency_bands_heatmap, plot_inference_time_comparison,
    plot_si_snr_comparison, plot_mfcc_distance_matrix,
    plot_spectral_features_comparison, plot_stems_energy_pie,
    plot_sdr_evolution_summary, plot_metrics_radar
)

print(f"✅ Imports OK — Device détecté par défaut : {'cuda' if __import__('torch').cuda.is_available() else 'cpu'}")
print(f"📂 Racine du projet : {PROJECT_ROOT}")

---
## 📂 2. Sélection du fichier audio test

Deux options :
- **Option A** : un fichier quelconque (MP3, WAV...) — uniquement les analyses sans vérité terrain
- **Option B** : un morceau issu de **MUSDB-HQ** — active toutes les métriques de qualité (SDR, SIR, SAR...)

> 📝 **Modifie `USE_MUSDB` et les chemins selon ton setup.**

In [ ]:
# ── Mode de travail ──────────────────────────────────────────────────────────
# Mettre USE_MUSDB = True si tu as une vérité terrain (pour SDR/SIR/SAR/ISR)
USE_MUSDB = False  # ← Changer ici

# ── Option A : fichier audio quelconque ──────────────────────────────────────
AUDIO_FILE = Path(PROJECT_ROOT) / 'data' / 'input' / 'good for the ghost - Alge.mp3'

# ── Option B : morceau MUSDB (avec vérité terrain) ───────────────────────────
# MUSDB_TRACK_DIR = Path(PROJECT_ROOT) / 'dataset' / 'musdb18hq' / 'test' / 'Arise - Run Run Run'
# AUDIO_FILE = MUSDB_TRACK_DIR / 'mixture.wav'

# ── Ground Truth (uniquement si USE_MUSDB = True) ────────────────────────────
# GT_STEMS = {
#     'vocals': MUSDB_TRACK_DIR / 'vocals.wav',
#     'drums':  MUSDB_TRACK_DIR / 'drums.wav',
#     'bass':   MUSDB_TRACK_DIR / 'bass.wav',
#     'other':  MUSDB_TRACK_DIR / 'other.wav',
# }

# ── Dossier de sortie ─────────────────────────────────────────────────────────
OUTPUT_DIR = Path(PROJECT_ROOT) / 'data' / 'output_demucs_comparison'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Modèles à comparer ────────────────────────────────────────────────────────
MODELS = SUPPORTED_MODELS  # ['htdemucs', 'htdemucs_ft', 'mdx_extra', 'mdx_extra_q']

# ── Vérification ─────────────────────────────────────────────────────────────
assert AUDIO_FILE.exists(), f"❌ Fichier introuvable : {AUDIO_FILE}"

duration = audio_utils.get_duration(AUDIO_FILE)
print(f"🎵 Fichier audio : {AUDIO_FILE.name}")
print(f"⏱️  Durée         : {duration:.1f} s ({duration/60:.1f} min)")
print(f"🤖 Modèles à comparer : {MODELS}")
print(f"💾 Sorties enregistrées dans : {OUTPUT_DIR}")
print(f"📊 Vérité terrain (MUSDB) : {'OUI' if USE_MUSDB else 'NON — métriques BSS désactivées'}")

In [ ]:
# ── Lecture du fichier audio source ──────────────────────────────────────────
display(Markdown("### 🔈 Écoute du fichier source"))
display(Audio(str(AUDIO_FILE), autoplay=False))

---
## ⏱️ 3. Séparation avec les 4 modèles — Benchmarking du temps d'inférence

> Cette cellule exécute la séparation complète avec chaque modèle en mesurant le temps.  
> **Attention** : selon le CPU/GPU, cela peut prendre plusieurs minutes par modèle.

In [ ]:
# ── Séparation (chargement du modèle + inférence) ─────────────────────────────
inference_times   = {}   # {model: temps_inférence_seule}
load_times        = {}   # {model: temps_chargement_modèle}
total_times       = {}   # {model: temps_total}
stem_paths        = {}   # {model: {stem: Path}}
stem_audio_arrays = {}   # {model: {stem: np.ndarray}} — chargé après séparation

for model_name in MODELS:
    print(f"\n{'='*60}")
    print(f"  🤖 Modèle : {MODEL_LABELS[model_name]}")
    print(f"{'='*60}")

    model_out_dir = OUTPUT_DIR / model_name
    model_out_dir.mkdir(parents=True, exist_ok=True)

    # ── Chargement du modèle ─────────────────────────────────────────────────
    with Timer() as t_load:
        separator = AudioSeparator(model_name=model_name)
    load_times[model_name] = t_load.elapsed
    print(f"  ⏳ Chargement du modèle : {t_load.elapsed:.2f}s")

    # ── Inférence ────────────────────────────────────────────────────────────
    with Timer() as t_inf:
        saved_paths = separator.process_file(AUDIO_FILE, model_out_dir)
    inference_times[model_name] = t_inf.elapsed
    total_times[model_name] = t_load.elapsed + t_inf.elapsed
    print(f"  ⚡ Inférence seule      : {t_inf.elapsed:.2f}s")
    print(f"  🏁 Total               : {total_times[model_name]:.2f}s")
    print(f"  📁 Fichiers générés    : {[p.name for p in saved_paths]}")

    # ── Indexation des stems ─────────────────────────────────────────────────
    stem_paths[model_name] = {p.stem.split('_')[-1]: p for p in saved_paths}
    # Pour les modèles dont le stem 'other' peut s'appeler autre chose
    # On suppose la convention : filename_{stem}.wav

print("\n✅ Séparation terminée pour tous les modèles !")

In [ ]:
# ── Chargement des stems en mémoire (numpy) ───────────────────────────────────
print("📥 Chargement des stems en mémoire...")
for model_name in MODELS:
    stem_audio_arrays[model_name] = {}
    for stem_name, path in stem_paths[model_name].items():
        audio, _ = audio_utils.load_audio(path, sr=DEFAULT_SAMPLE_RATE, mono=False)
        stem_audio_arrays[model_name][stem_name] = audio
    print(f"  ✓ {MODEL_LABELS[model_name]} — {list(stem_audio_arrays[model_name].keys())}")

print("\n✅ Données en mémoire !")
print(f"   Keys disponibles : modèles → stems → np.ndarray(channels, time)")

---
## ⏱️ 4. Visualisation des temps d'inférence

In [ ]:
fig = plot_inference_time_comparison(inference_times, duration)
plt.show()

# ── Tableau récapitulatif ─────────────────────────────────────────────────────
df_times = pd.DataFrame({
    'Modèle':         [MODEL_LABELS[m] for m in MODELS],
    'Chargement (s)': [round(load_times[m], 2) for m in MODELS],
    'Inférence (s)':  [round(inference_times[m], 2) for m in MODELS],
    'Total (s)':      [round(total_times[m], 2) for m in MODELS],
    'RTF':            [round(inference_times[m] / duration, 3) for m in MODELS],
}).set_index('Modèle')

display(Markdown("### 📋 Tableau des temps"))
display(df_times.style.background_gradient(cmap='RdYlGn_r', subset=['Inférence (s)', 'RTF']))

---
## 🔊 5. Écoute des stems séparés

In [ ]:
# On affiche les players audio pour chaque modèle × chaque stem
# (pratique pour une évaluation auditive subjective)

STEM_TO_DISPLAY = STEM_NAMES  # ['vocals', 'drums', 'bass', 'other']

for stem_name in STEM_TO_DISPLAY:
    display(Markdown(f"### 🎙️ Stem : **{stem_name.upper()}**"))
    for model_name in MODELS:
        path = stem_paths[model_name].get(stem_name)
        if path and path.exists():
            display(Markdown(f"**{MODEL_LABELS[model_name]}**"))
            display(Audio(str(path), autoplay=False))
    display(Markdown("---"))

---
## 🌊 6. Comparaison des formes d'onde (Waveforms)

> Un bon modèle devrait produire un signal propre, sans artefacts ni bruit résiduel.

In [ ]:
for stem_name in STEM_NAMES:
    fig = plot_waveforms_comparison(
        stem_audio_arrays,
        stem_name=stem_name,
        sr=DEFAULT_SAMPLE_RATE,
        max_seconds=15.0
    )
    plt.suptitle(f"Formes d'onde — {stem_name.capitalize()}", 
                 color='white', fontsize=15, y=1.01)
    plt.tight_layout()
    plt.show()

---
## 🔊 7. Spectrogrammes comparatifs

> Les spectrogrammes log-mel permettent de visualiser la répartition énergétique en fréquence et en temps.  
> Un spectre propre (peu de 'fantômes' dans les zones vides) indique une bonne séparation.

In [ ]:
for stem_name in STEM_NAMES:
    fig = plot_spectrograms_grid(
        stem_audio_arrays,
        stem_name=stem_name,
        sr=DEFAULT_SAMPLE_RATE
    )
    plt.show()

---
## 📐 8. Métriques spectrales avancées

### 8.1 Énergie RMS et distribution par stem

In [ ]:
# ── Calcul de l'énergie RMS par modèle et par stem ───────────────────────────
rms_per_model = {}

for model_name in MODELS:
    rms_per_model[model_name] = {}
    for stem_name, audio in stem_audio_arrays[model_name].items():
        rms_per_model[model_name][stem_name] = compute_rms_energy(audio)

# ── DataFrame récap ───────────────────────────────────────────────────────────
df_rms = pd.DataFrame(rms_per_model).T.round(5)
df_rms.index = [MODEL_LABELS[m] for m in df_rms.index]
display(Markdown("### 📋 Énergie RMS par stem"))
display(df_rms.style.background_gradient(cmap='Blues', axis=None))

# ── Pie charts de distribution ────────────────────────────────────────────────
fig, axes = plt.subplots(1, len(MODELS), figsize=(20, 5))
fig.patch.set_facecolor('#0d1117')
for ax, model_name in zip(axes, MODELS):
    rms = rms_per_model[model_name]
    stems_present = [s for s in STEM_NAMES if s in rms]
    vals = [rms[s] for s in stems_present]
    from src.music_separation.analysis_tools import STEM_COLORS
    colors = [STEM_COLORS.get(s, 'gray') for s in stems_present]
    wedges, texts, autotexts = ax.pie(
        vals, labels=[s.capitalize() for s in stems_present],
        colors=colors, autopct='%1.1f%%', startangle=90,
        wedgeprops=dict(edgecolor='white', linewidth=1.2)
    )
    for t in texts: t.set_color('white'); t.set_fontsize(10)
    for at in autotexts: at.set_color('black'); at.set_fontsize(9)
    ax.set_facecolor('#0d1117')
    ax.set_title(MODEL_LABELS[model_name], color='white', fontsize=11)

fig.suptitle("Distribution d'énergie RMS entre les stems", color='white', fontsize=14)
plt.tight_layout()
plt.show()

### 8.2 Caractéristiques spectrales (Centroïde, Roll-off, Flux)

In [ ]:
# ── Calcul des features spectrales ───────────────────────────────────────────
print("📐 Calcul des caractéristiques spectrales...")
spectral_features = {}

for model_name in MODELS:
    spectral_features[model_name] = {}
    for stem_name, audio in stem_audio_arrays[model_name].items():
        spectral_features[model_name][stem_name] = {
            'centroid': compute_spectral_centroid(audio, sr=DEFAULT_SAMPLE_RATE),
            'rolloff':  compute_spectral_rolloff(audio, sr=DEFAULT_SAMPLE_RATE),
            'flux':     compute_spectral_flux(audio, sr=DEFAULT_SAMPLE_RATE),
        }
    print(f"  ✓ {MODEL_LABELS[model_name]}")

fig = plot_spectral_features_comparison(spectral_features, STEM_NAMES)
plt.show()

# Tableau recap centroïde
df_centroid = pd.DataFrame({
    model_name: {stem: round(spectral_features[model_name][stem]['centroid'], 1)
                 for stem in STEM_NAMES if stem in spectral_features[model_name]}
    for model_name in MODELS
}).T.rename(index=MODEL_LABELS)
display(Markdown("### 📋 Centroïde spectral (Hz) — plus élevé = son plus brillant"))
display(df_centroid.style.background_gradient(cmap='cool', axis=None))

### 8.3 Énergie par bandes de fréquences

In [ ]:
print("📊 Calcul de l'énergie par bandes...")
band_energies = {}  # {model: {stem: {band: value}}}

for model_name in MODELS:
    band_energies[model_name] = {}
    for stem_name, audio in stem_audio_arrays[model_name].items():
        band_energies[model_name][stem_name] = compute_frequency_band_energy(
            audio, sr=DEFAULT_SAMPLE_RATE
        )
    print(f"  ✓ {MODEL_LABELS[model_name]}")

# ── Heatmap par stem ──────────────────────────────────────────────────────────
for stem_name in STEM_NAMES:
    if all(stem_name in band_energies[m] for m in MODELS):
        fig = plot_frequency_bands_heatmap(band_energies, stem_name=stem_name)
        plt.show()

### 8.4 Distance MFCC — Similarité timbrale

> La distance MFCC mesure à quel point le timbre reconstruit ressemble à une référence.  
> Ici on compare chaque modèle à **htdemucs** (pris comme référence interne).
> Une distance faible = reconstruction similaire au modèle de référence.

In [ ]:
print("🎼 Calcul des distances MFCC (référence = htdemucs)...")
ref_model = 'htdemucs'
mfcc_distances = {}  # {model: {stem: distance}}

for model_name in MODELS:
    mfcc_distances[model_name] = {}
    for stem_name in STEM_NAMES:
        ref_audio = stem_audio_arrays.get(ref_model, {}).get(stem_name)
        est_audio = stem_audio_arrays.get(model_name, {}).get(stem_name)
        if ref_audio is not None and est_audio is not None:
            dist = compute_mfcc_distance(ref_audio, est_audio, sr=DEFAULT_SAMPLE_RATE)
            mfcc_distances[model_name][stem_name] = dist
    print(f"  ✓ {MODEL_LABELS[model_name]}")

fig = plot_mfcc_distance_matrix(mfcc_distances, stem_names=STEM_NAMES)
plt.show()

df_mfcc = pd.DataFrame(mfcc_distances).T.round(2).rename(index=MODEL_LABELS)
display(Markdown(f"### 📋 Distance MFCC vs {MODEL_LABELS[ref_model]} (↓ = plus similaire)"))
display(df_mfcc.style.background_gradient(cmap='RdYlGn_r', axis=None))

---
## ⚡ 9. SI-SNR (Scale-Invariant SNR)

> Le SI-SNR est invariant au gain et mesure la qualité de séparation  
> en comparant les stems entre modèles **(référence = htdemucs)**.

In [ ]:
print("⚡ Calcul des SI-SNR (référence interne = htdemucs)...")
ref_model = 'htdemucs'
si_snr_per_model = {}  # {model: {stem: si_snr}}
snr_per_model    = {}  # {model: {stem: snr}}

for model_name in MODELS:
    si_snr_per_model[model_name] = {}
    snr_per_model[model_name] = {}
    for stem_name in STEM_NAMES:
        ref = stem_audio_arrays.get(ref_model, {}).get(stem_name)
        est = stem_audio_arrays.get(model_name, {}).get(stem_name)
        if ref is not None and est is not None:
            si_snr_per_model[model_name][stem_name] = compute_si_snr(ref, est)
            snr_per_model[model_name][stem_name]    = compute_snr(ref, est)
    print(f"  ✓ {MODEL_LABELS[model_name]}")

fig = plot_si_snr_comparison(si_snr_per_model, stem_names=STEM_NAMES)
plt.show()

# Tableau
df_sisnr = pd.DataFrame(si_snr_per_model).T.round(2).rename(index=MODEL_LABELS)
display(Markdown("### 📋 SI-SNR par stem (dB) — ↑ meilleur"))
display(df_sisnr.style
        .background_gradient(cmap='RdYlGn', axis=None)
        .highlight_max(color='#1a472a', axis=0)
        .highlight_min(color='#7d1111', axis=0))

---
## 📊 10. Métriques BSS officielles (SDR / SIR / SAR / ISR)

> ⚠️ Cette section nécessite la **vérité terrain** — n'est active que si `USE_MUSDB = True`.
> Si `USE_MUSDB = False`, on affiche un message informatif et on passe à la suite.

In [ ]:
bss_results = {}   # {model: {metric: np.ndarray per stem}}
HAS_BSS = False

if not USE_MUSDB:
    display(Markdown(
        "> ℹ️ **Mode sans vérité terrain** : les métriques BSS (SDR/SIR/SAR/ISR) sont désactivées.  \n"
        "> Passez `USE_MUSDB = True` et spécifiez `GT_STEMS` pour activer ces mesures."
    ))
else:
    evaluator = AudioEvaluator(sample_rate=DEFAULT_SAMPLE_RATE)
    
    # Prépare les chemins GT dans l'ordre des sources
    gt_paths = [GT_STEMS[stem] for stem in STEM_NAMES]

    for model_name in MODELS:
        print(f"\n📊 Évaluation BSS : {MODEL_LABELS[model_name]}...")
        pred_paths = [stem_paths[model_name].get(stem) for stem in STEM_NAMES]
        pred_paths = [p for p in pred_paths if p is not None]
        
        try:
            metrics = evaluator.compute_bss_metrics(gt_paths, pred_paths)
            bss_results[model_name] = metrics
            print(f"  SDR : {np.round(metrics['SDR'], 2)}")
            if 'ISR' in metrics:
                print(f"  ISR : {np.round(metrics['ISR'], 2)}")
            print(f"  SIR : {np.round(metrics['SIR'], 2)}")
            print(f"  SAR : {np.round(metrics['SAR'], 2)}")
        except Exception as e:
            print(f"  ❌ Erreur : {e}")
    
    HAS_BSS = len(bss_results) > 0
    print("\n✅ Évaluation BSS terminée !")

In [ ]:
if HAS_BSS:
    # ── Graphiques barres groupées ────────────────────────────────────────────
    for metric_name in ['SDR', 'SIR', 'SAR', 'ISR']:
        # Vérifie si la métrique est disponible
        if any(metric_name in bss_results[m] for m in bss_results):
            fig = plot_metrics_bar_per_stem(
                bss_results, metric_name=metric_name, stem_names=STEM_NAMES
            )
            plt.show()

    # ── Vue synthèse ─────────────────────────────────────────────────────────
    fig = plot_sdr_evolution_summary(bss_results, stem_names=STEM_NAMES)
    plt.show()

    # ── Tableau récap SDR ─────────────────────────────────────────────────────
    df_sdr = pd.DataFrame({
        MODEL_LABELS[m]: {
            f"{s.capitalize()} SDR": round(float(bss_results[m]['SDR'][i]), 2)
            for i, s in enumerate(STEM_NAMES)
            if i < len(bss_results[m]['SDR'])
        }
        for m in bss_results
    })
    display(Markdown("### 📋 SDR par stem et par modèle"))
    display(df_sdr.style
            .background_gradient(cmap='RdYlGn', axis=1)
            .highlight_max(color='#1a472a', axis=1)
            .highlight_min(color='#7d1111', axis=1))

---
## 🕸️ 11. Vue Synthèse — Graphique Radar

> Comparaison globale sur un ensemble de métriques normalisées.

In [ ]:
# ── Construction du dictionnaire de métriques normalisées pour le radar ───────
# On normalise chaque métrique [0, 1] pour les rendre comparables.
# Métriques : SI-SNR moyen, centroïde moyen voix, bas RTF, énergie basse bass

def norm(values: dict, higher_is_better: bool = True) -> dict:
    """Normalise les valeurs dict {model: val} entre 0 et 1."""
    vals = np.array(list(values.values()), dtype=float)
    vmin, vmax = np.nanmin(vals), np.nanmax(vals)
    if vmax == vmin:
        return {m: 0.5 for m in values}
    normed = (vals - vmin) / (vmax - vmin)
    if not higher_is_better:
        normed = 1 - normed
    return {m: float(normed[i]) for i, m in enumerate(values)}

# ── Collecte des métriques scalaires ──────────────────────────────────────────
radar_metrics = {}

# 1) SI-SNR global moyen (tous stems, versus htdemucs)
sisnr_global = {m: np.nanmean(list(si_snr_per_model[m].values())) for m in MODELS}
sisnr_normed = norm(sisnr_global, higher_is_better=True)

# 2) Vitesse (RTF inversé → plus c'est bas, mieux c'est)
rtf_normed = norm(inference_times, higher_is_better=False)

# 3) Stabilité spectrale (flux inversé — moins élevé = plus propre)
flux_global = {
    m: np.nanmean([spectral_features[m][s]['flux'] for s in STEM_NAMES if s in spectral_features[m]])
    for m in MODELS
}
flux_normed = norm(flux_global, higher_is_better=False)

# 4) Séparation basse : énergie relative dans la bande bass pour le stem 'bass'
bass_band_key = 'bass (80-250 Hz)'
bass_energy = {
    m: band_energies[m].get('bass', {}).get(bass_band_key, 0.0)
    for m in MODELS
}
bass_normed = norm(bass_energy, higher_is_better=True)

# 5) Clarté voix : énergie dans mid pour le stem 'vocals'
mid_key = 'midrange (250-2kHz)'
vocals_mid = {
    m: band_energies[m].get('vocals', {}).get(mid_key, 0.0)
    for m in MODELS
}
vocals_normed = norm(vocals_mid, higher_is_better=True)

# 6) Similarité timbrale globale (MFCC distance inversée vs htdemucs)
mfcc_global = {
    m: np.nanmean(list(mfcc_distances[m].values())) if mfcc_distances[m] else 0.0
    for m in MODELS
}
mfcc_normed = norm(mfcc_global, higher_is_better=False)

# ── Assemblage pour le radar ──────────────────────────────────────────────────
for m in MODELS:
    radar_metrics[m] = {
        'SI-SNR':          sisnr_normed[m],
        'Vitesse':         rtf_normed[m],
        'Propreté\nspectrale': flux_normed[m],
        'Clarté\nbasses':  bass_normed[m],
        'Clarté\nvoix':    vocals_normed[m],
        'Similarité\ntimbrale': mfcc_normed[m],
    }

fig = plot_metrics_radar(radar_metrics, title="Comparaison radar (normalisée 0–1)")
plt.show()

# Tableau recap
df_radar = pd.DataFrame(radar_metrics).T.round(3).rename(index=MODEL_LABELS)
display(Markdown("### 📋 Scores normalisés (0 = pire, 1 = meilleur)"))
display(df_radar.style.background_gradient(cmap='RdYlGn', axis=None)
        .highlight_max(color='#1a472a', axis=0)
        .highlight_min(color='#7d1111', axis=0))

---
## 📊 12. Récapitulatif final — Tableau des scores globaux

In [ ]:
# ── Score composite ───────────────────────────────────────────────────────────
# Calcul d'un score final pondéré

weights = {
    'SI-SNR':               0.25,   # Qualité de séparation
    'Vitesse':              0.20,   # Performance computationnelle
    'Propreté\nspectrale':  0.15,   # Artefacts spectraux
    'Clarté\nbasses':       0.15,   # Qualité de la séparation basse
    'Clarté\nvoix':         0.10,   # Qualité de la séparation voix
    'Similarité\ntimbrale': 0.15,   # Fidélité timbrale
}

final_scores = {}
for m in MODELS:
    score = sum(radar_metrics[m][metric] * w for metric, w in weights.items())
    final_scores[m] = round(score, 4)

# Tri par score décroissant
ranked = sorted(final_scores.items(), key=lambda x: x[1], reverse=True)

print("\n" + "="*60)
print("  🏆 CLASSEMENT FINAL DES MODÈLES")
print("="*60)
medals = ['🥇', '🥈', '🥉', '4️⃣']
for i, (model, score) in enumerate(ranked):
    print(f"  {medals[i]} {MODEL_LABELS[model]:<20} → Score : {score:.4f}")
print("="*60)

# ── Barres finales ────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#0d1117')

sorted_models = [m for m, _ in ranked]
scores = [final_scores[m] for m in sorted_models]
colors = [MODEL_COLORS[m] for m in sorted_models]
labels = [MODEL_LABELS[m] for m in sorted_models]

bars = ax.barh(labels[::-1], scores[::-1], color=colors[::-1], 
               edgecolor='white', linewidth=0.5, alpha=0.88)
for bar, s in zip(bars, scores[::-1]):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height() / 2,
            f"{s:.3f}", va='center', ha='left', color='white', fontsize=11, fontweight='bold')

ax.set_xlabel('Score composite pondéré', color='white', fontsize=12)
ax.set_title('🏆 Classement Final des Modèles Demucs', color='white', fontsize=14)
ax.tick_params(colors='white')
ax.spines[:].set_color('#333')
ax.xaxis.grid(True, color='#333', linestyle='--', linewidth=0.8)
ax.set_xlim(0, max(scores) * 1.15)
plt.tight_layout()
plt.show()

---
## 🏆 13. Verdict Final & Analyse


In [ ]:
best_model = ranked[0][0]
fastest_model = min(inference_times, key=inference_times.get)
most_timbrally_similar = min(
    [m for m in MODELS if m != 'htdemucs'],
    key=lambda m: np.nanmean(list(mfcc_distances[m].values())) if mfcc_distances[m] else float('inf'),
    default=ranked[1][0]
)

verdict = f"""
## 🎯 Verdict Final

### 🏆 Meilleur modèle global
**{MODEL_LABELS[best_model]}** remporte la comparaison avec un score composite de **{final_scores[best_model]:.3f}**.

### ⚡ Modèle le plus rapide
**{MODEL_LABELS[fastest_model]}** est le plus rapide avec **{inference_times[fastest_model]:.1f}s** d'inférence  
(RTF = {inference_times[fastest_model]/duration:.2f}×).

### 🎼 Fidélité timbrale la plus proche de HTDemucs
**{MODEL_LABELS[most_timbrally_similar]}** produit les stems les plus similaires à HTDemucs dans l'espace MFCC.

### 📊 Classement complet
| Rang | Modèle | Score |
|------|--------|-------|
""" + "\n".join([f"| {i+1} | {MODEL_LABELS[m]} | {s:.3f} |" for i, (m, s) in enumerate(ranked)])

verdict += f"""

### 💡 Recommandations par usage

| Contexte | Modèle recommandé | Raison |
|----------|------------------|--------|
| **Production musicale** (qualité max) | {MODEL_LABELS[best_model]} | Meilleur score global |
| **Temps réel / edge computing** | {MODEL_LABELS[fastest_model]} | Le plus rapide |
| **Équilibre qualité/vitesse** | HTDemucs | Architecture Hybrid Transformer robuste |
| **Ressources GPU limitées** | MDX Extra Q | Modèle quantifié, plus léger |
"""

display(Markdown(verdict))

In [ ]:
# ── Export des résultats en CSV ───────────────────────────────────────────────
results_dir = OUTPUT_DIR / 'results'
results_dir.mkdir(parents=True, exist_ok=True)

# Temps
df_times.to_csv(results_dir / 'inference_times.csv')

# Scores normalisés
df_radar.to_csv(results_dir / 'normalized_scores.csv')

# RMS
df_rms.to_csv(results_dir / 'rms_energy.csv')

# MFCC
df_mfcc.to_csv(results_dir / 'mfcc_distances.csv')

# SI-SNR
df_sisnr.to_csv(results_dir / 'si_snr.csv')

# Score final
df_final = pd.DataFrame(list(final_scores.items()), columns=['model', 'score'])
df_final['model'] = df_final['model'].map(MODEL_LABELS)
df_final = df_final.sort_values('score', ascending=False).set_index('model')
df_final.to_csv(results_dir / 'final_scores.csv')

print(f"✅ Résultats exportés dans : {results_dir}")
print(f"   Fichiers : {[f.name for f in results_dir.glob('*.csv')]}")

---
> **Notebook réalisé dans le cadre du Projet IA — ENSTA Paris 2025/2026**  
> Modèles testés : `htdemucs`, `htdemucs_ft`, `mdx_extra`, `mdx_extra_q`  
> Framework : [Demucs](https://github.com/facebookresearch/demucs) (Meta AI Research)